# 作业

## 1. 一阶马尔可夫模型的条件概率估计（拉普拉斯平滑）

给定字符序列 "ababc"，词汇表 V = {'a','b','c'}，采用一阶马尔可夫模型，使用加1平滑估计条件概率。

**统计转移频次（真实序列中）：**

从 'b' 出发的转移：
- 'b' → 'a'：出现 2 次（位置2→3，位置4→5）
- 'b' → 'c'：出现 0 次
- 'b' → 'b'：出现 0 次

从 'b' 出发的总次数 N(b) = 2。

**加1平滑公式：**
P(w|b) = (count(b, w) + 1) / (N(b) + |V|)

**计算：**
P('a'|'b') = (2 + 1) / (2 + 3) = 3/5 = 0.6

P('c'|'b') = (0 + 1) / (2 + 3) = 1/5 = 0.2

---

## 2. 线性RNN梯度推导与消失/爆炸条件

**模型定义：**
h_t = W_hh * h_{t-1} + W_hx * x_t

o_t = W_oh * h_t

L = (1/2) * Σ_{t=1}^T (o_t - y_t)^2

**对 W_hh 的梯度（通过时间反向传播）：**

损失对 W_hh 的梯度为各时间步贡献之和：

∂L/∂W_hh = Σ_{t=1}^T Σ_{k=1}^t (∂L/∂o_t) * (∂o_t/∂h_t) * (∂h_t/∂h_k) * (∂h_k/∂W_hh)

其中，从第 k 步到第 t 步的 Jacobian 为：

∂h_t/∂h_k = Π_{i=k+1}^t W_hh （当 t > k 时）

因此梯度展开式为：

∂L/∂W_hh = Σ_{t=1}^T (o_t - y_t) * W_oh^T * Σ_{k=1}^t (W_hh)^(t-k) * h_{k-1}^T

**梯度消失或爆炸条件：**

若 W_hh 的谱半径（最大特征值绝对值）> 1，则 (W_hh)^(t-k) 随间隔增长呈指数放大，导致梯度爆炸。

若 W_hh 的谱半径 < 1，则 (W_hh)^(t-k) 随间隔增长呈指数衰减，导致梯度消失。

若谱半径 = 1，梯度可稳定传播。

---

## 3. 深度双向RNN参数数量

**模型结构：**
- L 层，每层双向，每层每个方向隐藏单元数 H
- 输入维度 D，输出维度 O
- 每层包含前向和后向两个RNN单元

**参数计算：**

第1层（输入到隐藏）：
- 前向：W_hx (H × D) 和偏置 b_h (H) → HD + H
- 后向：同样 HD + H
- 小计：2(HD + H)

第 l 层（l ≥ 2，隐藏到隐藏）：
- 前向：W_hh (H × 2H) 和偏置 b_h (H) → 2H^2 + H
  （因为输入来自上一层前向和后向的拼接，维度 2H）
- 后向：同样 2H^2 + H
- 小计：2(2H^2 + H)

输出层（仅最后一层输出到输出）：
- W_oh (O × 2H) 和偏置 b_o (O) → 2HO + O

**总参数量：**

Total = 2(HD + H) + (L-1)*2(2H^2 + H) + (2HO + O)

化简：

Total = 2HD + 2H + 4(L-1)H^2 + 2(L-1)H + 2HO + O

Total = 4(L-1)H^2 + 2HD + 2HO + 2LH + O

---

## 4. Skip-gram 负采样损失函数

**给定：** 中心词 w_c，上下文词 w_o，负样本集合 {w_k | k = 1, 2, ..., K}

**负采样目标函数（对数似然形式，最大化）：**

J = log σ(v_c · u_o) + Σ_{k=1}^K E_{w_k ~ P_n(w)} [ log σ(-v_c · u_k) ]

**等价地，常用损失函数（最小化负对数似然）：**

L = - log σ(v_c^T u_o) - Σ_{k=1}^K log σ(-v_c^T u_k)

其中 σ(x) = 1/(1 + e^(-x)) 是 sigmoid 函数。

**负样本采样：**

从噪声分布 P_n(w) 中独立抽取 K 个负样本。

P_n(w) 通常取词频的 3/4 次方分布：

P_n(w) ∝ count(w)^(3/4) / Σ_v count(v)^(3/4)

这样能降低高频词的采样概率，提高低频词被采到的机会。

**完整目标函数（以损失形式）：**

L = - log( 1 / (1 + e^(-v_c^T u_o)) ) - Σ_{k=1}^K log( 1 / (1 + e^(v_c^T u_k)) )

即：

L = - log σ(v_c^T u_o) - Σ_{k=1}^K log σ(-v_c^T u_k)

---

## 5. 缩放点积注意力计算

**给定：**

Q ∈ R^(2×4), K ∈ R^(3×4), V ∈ R^(3×5), d_k = 4

**步骤1：计算得分矩阵 S = Q K^T / √d_k**

Q K^T 为 2×3 矩阵，每个元素为 Q 的行与 K 的行的点积。

设 Q 的第 i 行为 q_i，K 的第 j 行为 k_j：

S_ij = (q_i · k_j) / 2

所以：

S = (1/2) * [q1·k1, q1·k2, q1·k3; q2·k1, q2·k2, q2·k3]

**步骤2：对 S 的每一行应用 softmax（按行）**

A_ij = exp(S_ij) / Σ_{m=1}^3 exp(S_im)

得到注意力权重矩阵 A ∈ R^(2×3)，每行和为 1。

**步骤3：加权求和得到输出 O = A V**

O_i = Σ_{j=1}^3 A_ij * V_j

其中 V_j 表示 V 的第 j 行（维度为 1×5）。

**最终输出矩阵 O ∈ R^(2×5)。**

（具体数值需代入 Q, K, V 的实际值计算，此处给出计算流程框架。）

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    对文本进行预处理，构建词汇表，并生成自回归语言模型的特征序列和标签。

    参数:
        text (str): 输入文本
        n (int): 滑动窗口长度（特征词个数）

    返回:
        dict: 词汇表字典 {词: ID}
        list: 特征列表，每个特征是一个长度为 n 的词ID列表
        list: 标签列表，每个标签是下一个词的ID（若不存在则忽略）
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    # 使用正则表达式保留字母和空格，去掉其他字符
    text_clean = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    
    # 2. 按空格分词（多个空格会被 split 自动处理）
    words = text_clean.split()
    
    if not words:
        return {}, [], []
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(words)
    # 按频率降序排序，频率相同则按字母顺序（保证确定性）
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 将词转换为ID序列
    ids = [vocab[word] for word in words]
    
    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    
    for i in range(len(ids) - n):
        # 特征：当前窗口的 n 个词 ID
        feature = ids[i:i+n]
        # 标签：窗口后的下一个词 ID
        label = ids[i+n]
        features.append(feature)
        labels.append(label)
    
    return vocab, features, labels


# 示例测试（使用题目给出的输入）
if __name__ == "__main__":
    text = "The time machine"
    n = 2
    vocab, features, labels = preprocess_text(text, n)
    
    print("词汇表:", vocab)
    print("特征列表:", features)
    print("标签列表:", labels)
    
    # 将 ID 转换回词以便查看（仅作验证）
    id_to_word = {v: k for k, v in vocab.items()}
    features_words = [[id_to_word[id] for id in feat] for feat in features]
    labels_words = [id_to_word[label] if label is not None else None for label in labels]
    print("特征（词形式）:", features_words)
    print("标签（词形式）:", labels_words)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征列表: [[1, 2]]
标签列表: [0]
特征（词形式）: [['the', 'time']]
标签（词形式）: ['machine']


In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN 单元前向传播
    
    参数:
        x_t: 当前输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏的权重，形状 (input_size, hidden_size)
        W_hh: 隐藏到隐藏的权重，形状 (hidden_size, hidden_size)
        b_h: 偏置，形状 (hidden_size,)
    
    返回:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存用于反向传播的中间变量
    """
    # 计算线性变换
    # h_linear = x_t @ W_hx + h_prev @ W_hh + b_h
    # 形状: (batch_size, hidden_size)
    h_linear = np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h
    
    # 应用 tanh 激活函数
    h_t = np.tanh(h_linear)
    
    # 缓存中间变量用于反向传播
    cache = (x_t, h_prev, W_hx, W_hh, b_h, h_linear, h_t)
    
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    RNN 单元反向传播（单步）
    
    参数:
        dh_next: 损失对 h_t 的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播缓存的中间变量
    
    返回:
        dx_t: 损失对 x_t 的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对 h_prev 的梯度，形状 (batch_size, hidden_size)
        dW_hx: 损失对 W_hx 的梯度，形状 (input_size, hidden_size)
        dW_hh: 损失对 W_hh 的梯度，形状 (hidden_size, hidden_size)
        db_h: 损失对 b_h 的梯度，形状 (hidden_size,)
    """
    # 解包缓存
    x_t, h_prev, W_hx, W_hh, b_h, h_linear, h_t = cache
    
    batch_size = x_t.shape[0]
    hidden_size = h_t.shape[1]
    
    # 1. 计算 tanh 激活函数的梯度
    # dtanh/dh_linear = 1 - tanh^2 = 1 - h_t^2
    dtanh = 1 - h_t ** 2  # 形状: (batch_size, hidden_size)
    
    # 2. 损失对 h_linear 的梯度
    # dh_linear = dh_next * dtanh (链式法则)
    dh_linear = dh_next * dtanh  # 形状: (batch_size, hidden_size)
    
    # 3. 计算损失对参数的梯度
    # dW_hx = x_t^T @ dh_linear
    dW_hx = np.dot(x_t.T, dh_linear)  # 形状: (input_size, hidden_size)
    
    # dW_hh = h_prev^T @ dh_linear
    dW_hh = np.dot(h_prev.T, dh_linear)  # 形状: (hidden_size, hidden_size)
    
    # db_h = sum(dh_linear, axis=0) (对 batch 维度求和)
    db_h = np.sum(dh_linear, axis=0)  # 形状: (hidden_size,)
    
    # 4. 计算损失对输入的梯度
    # dx_t = dh_linear @ W_hx^T
    dx_t = np.dot(dh_linear, W_hx.T)  # 形状: (batch_size, input_size)
    
    # dh_prev = dh_linear @ W_hh^T
    dh_prev = np.dot(dh_linear, W_hh.T)  # 形状: (batch_size, hidden_size)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# ============ 测试代码 ============
if __name__ == "__main__":
    # 设置随机种子以便复现
    np.random.seed(42)
    
    # 定义维度
    batch_size = 3
    input_size = 4
    hidden_size = 5
    
    # 随机初始化输入和参数
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)
    
    print("=== 前向传播 ===")
    h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print(f"x_t 形状: {x_t.shape}")
    print(f"h_prev 形状: {h_prev.shape}")
    print(f"h_t 形状: {h_t.shape}")
    print(f"h_t 前几个值:\n{h_t[:2, :3]}")
    
    # 模拟上游梯度（来自损失函数）
    dh_next = np.random.randn(batch_size, hidden_size)
    
    print("\n=== 反向传播 ===")
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)
    
    print(f"dx_t 形状: {dx_t.shape}")
    print(f"dh_prev 形状: {dh_prev.shape}")
    print(f"dW_hx 形状: {dW_hx.shape}")
    print(f"dW_hh 形状: {dW_hh.shape}")
    print(f"db_h 形状: {db_h.shape}")
    
    print(f"\ndx_t 前几个值:\n{dx_t[:2, :3]}")
    print(f"dh_prev 前几个值:\n{dh_prev[:2, :3]}")
    print(f"dW_hx 前几个值:\n{dW_hx[:3, :3]}")
    print(f"dW_hh 前几个值:\n{dW_hh[:3, :3]}")
    print(f"db_h 前几个值:\n{db_h[:3]}")
    
    # ============ 数值梯度验证 ============
    print("\n=== 数值梯度验证（使用有限差分法）===")
    
    def numerical_gradient(func, params, delta=1e-5):
        """计算数值梯度以验证反向传播"""
        grads = []
        for i, param in enumerate(params):
            param_flat = param.flatten()
            grad_flat = np.zeros_like(param_flat)
            
            for j in range(len(param_flat)):
                # 正向扰动
                param_flat[j] += delta
                params[i] = param_flat.reshape(param.shape)
                loss_plus = func()
                
                # 负向扰动
                param_flat[j] -= 2 * delta
                params[i] = param_flat.reshape(param.shape)
                loss_minus = func()
                
                # 还原
                param_flat[j] += delta
                params[i] = param_flat.reshape(param.shape)
                
                # 中心差分
                grad_flat[j] = (loss_plus - loss_minus) / (2 * delta)
            
            grads.append(grad_flat.reshape(param.shape))
        return grads
    
    # 定义损失函数（简单的标量损失，用于数值验证）
    def compute_loss(params):
        """根据参数计算标量损失"""
        # 重新计算前向传播
        h_t, _ = rnn_cell_forward(x_t, h_prev, params[0], params[1], params[2])
        # 模拟一个简单的损失：输出 h_t 的平方和
        return np.sum(h_t ** 2)
    
    # 保存原始参数
    params_orig = [W_hx.copy(), W_hh.copy(), b_h.copy()]
    
    # 构建用于数值梯度的参数列表（需要梯度计算的参数）
    params_for_grad = [W_hx.copy(), W_hh.copy(), b_h.copy()]
    
    # 计算数值梯度
    num_grads = numerical_gradient(
        lambda: compute_loss(params_for_grad),
        params_for_grad
    )
    
    # 计算解析梯度（从反向传播得到）
    # 对于损失 L = sum(h_t^2)，上游梯度 dh_next = 2 * h_t
    dh_next_analytical = 2 * h_t
    _, _, dW_hx_analytical, dW_hh_analytical, db_h_analytical = rnn_cell_backward(
        dh_next_analytical, cache
    )
    
    # 比较数值梯度和解析梯度
    print("W_hx 梯度差异:", np.max(np.abs(num_grads[0] - dW_hx_analytical)))
    print("W_hh 梯度差异:", np.max(np.abs(num_grads[1] - dW_hh_analytical)))
    print("b_h 梯度差异:", np.max(np.abs(num_grads[2] - db_h_analytical)))
    
    # 如果差异很小，说明反向传播正确
    if np.max(np.abs(num_grads[0] - dW_hx_analytical)) < 1e-6 and \
       np.max(np.abs(num_grads[1] - dW_hh_analytical)) < 1e-6 and \
       np.max(np.abs(num_grads[2] - db_h_analytical)) < 1e-6:
        print("\n✅ 梯度验证通过！反向传播实现正确。")
    else:
        print("\n❌ 梯度验证失败，请检查实现。")

=== 前向传播 ===
x_t 形状: (3, 4)
h_prev 形状: (3, 5)
h_t 形状: (3, 5)
h_t 前几个值:
[[ 0.37757047 -0.85065863 -0.99999996]
 [-0.99893487 -0.99615252 -0.99992555]]

=== 反向传播 ===
dx_t 形状: (3, 4)
dh_prev 形状: (3, 5)
dW_hx 形状: (4, 5)
dW_hh 形状: (5, 5)
db_h 形状: (5,)

dx_t 前几个值:
[[0.31738592 0.04540948 0.49436794]
 [1.69859152 0.19660404 0.15494235]]
dh_prev 前几个值:
[[-0.35500842 -0.01693622  0.05423127]
 [-0.35045347 -0.77344493 -0.17164285]]
dW_hx 前几个值:
[[-0.22974526  0.21780371 -0.00407317]
 [ 0.15219802 -0.23864969  0.00476795]
 [-0.26143473  0.21177928 -0.00423857]]
dW_hh 前几个值:
[[-4.64231031e-02 -2.44968913e-02  5.52054090e-04]
 [ 1.79155353e-01  5.75018951e-01 -1.23368542e-02]
 [ 3.19911118e-01  1.98744397e-01 -4.58539827e-03]]
db_h 前几个值:
[-0.03669355 -0.41373297  0.00861537]

=== 数值梯度验证（使用有限差分法）===
W_hx 梯度差异: 2.196680615185187e-10
W_hh 梯度差异: 6.758122950145662e-10
b_h 梯度差异: 1.3091772110840338e-10

✅ 梯度验证通过！反向传播实现正确。


In [1]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    """
    双向RNN编码器（使用 torch.nn.RNN）
    
    参数:
        input_dim: 输入特征维度
        hidden_dim: 隐藏状态维度（每个方向）
        num_layers: RNN层数（默认为1）
        dropout: dropout概率（仅当num_layers > 1时有效）
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super(BidirectionalRNNEncoder, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 双向RNN
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False,  # 输入形状: (seq_len, batch, input_dim)
            dropout=dropout if num_layers > 1 else 0.0
        )
    
    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回:
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        # 前向传播
        # outputs: (seq_len, batch, 2*hidden_dim)
        # h_n: (2*num_layers, batch, hidden_dim)
        outputs, h_n = self.rnn(X)
        
        # 获取最后一层的双向隐藏状态
        # h_n[-2] 是最后一层前向的最终状态
        # h_n[-1] 是最后一层后向的最终状态
        forward_last = h_n[-2, :, :]  # (batch, hidden_dim)
        backward_last = h_n[-1, :, :]  # (batch, hidden_dim)
        
        # 拼接前向和后向的最终状态
        final_state = torch.cat([forward_last, backward_last], dim=1)  # (batch, 2*hidden_dim)
        
        return outputs, final_state


# ============ 测试方式一 ============
if __name__ == "__main__":
    # 设置随机种子
    torch.manual_seed(42)
    
    # 参数设置
    seq_len = 5
    batch = 3
    input_dim = 10
    hidden_dim = 8
    
    # 创建输入
    X = torch.randn(seq_len, batch, input_dim)
    
    # 创建编码器
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers=1)
    
    # 前向传播
    outputs, final_state = encoder(X)
    
    print("=== 使用 torch.nn.RNN ===")
    print(f"输入形状: {X.shape}")
    print(f"输出形状: {outputs.shape}  # (seq_len, batch, 2*hidden_dim)")
    print(f"最终状态形状: {final_state.shape}  # (batch, 2*hidden_dim)")
    print(f"输出前几个值:\n{outputs[0, :, :3]}")
    print(f"最终状态前几个值:\n{final_state[:2, :3]}")

=== 使用 torch.nn.RNN ===
输入形状: torch.Size([5, 3, 10])
输出形状: torch.Size([5, 3, 16])  # (seq_len, batch, 2*hidden_dim)
最终状态形状: torch.Size([3, 16])  # (batch, 2*hidden_dim)
输出前几个值:
tensor([[ 0.7094,  0.7854, -0.7885],
        [ 0.4425,  0.3287, -0.1944],
        [ 0.2458,  0.3886, -0.3790]], grad_fn=<SliceBackward0>)
最终状态前几个值:
tensor([[-0.5749,  0.6340, -0.8587],
        [ 0.5035, -0.1934, -0.3447]], grad_fn=<SliceBackward0>)


In [3]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, W, W_out):
    """
    CBOW模型前向传播和损失计算
    
    Args:
        context_indices: 上下文词索引列表，形状 (batch_size, context_size)
        W: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    Returns:
        loss: 交叉熵损失值
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 获取上下文词的嵌入向量
    # context_indices: (batch_size, context_size)
    # W[context_indices]: (batch_size, context_size, d)
    context_embeddings = W[context_indices]  # (batch_size, context_size, d)
    
    # 计算平均上下文向量作为隐藏层
    # h: (batch_size, d)
    h = torch.mean(context_embeddings, dim=1)  # 对context_size维度求平均
    
    # 计算输出分数
    # scores: (batch_size, V)
    scores = h @ W_out  # (batch_size, d) @ (d, V) = (batch_size, V)
    
    # 计算输出概率分布 (softmax)
    probs = F.softmax(scores, dim=1)  # (batch_size, V)
    
    # 目标为中心词索引 (这里假设目标就是第一个上下文词，仅用于演示)
    # 实际使用中，目标词是单独传入的
    # 这里我们使用第一个上下文词作为目标进行演示
    targets = context_indices[:, 0]  # (batch_size,)
    
    # 计算交叉熵损失
    # 使用负对数似然: -log(probs[target])
    loss = F.cross_entropy(scores, targets)
    
    return loss

# 测试代码
if __name__ == "__main__":
    torch.manual_seed(42)
    
    V = 10  # 词汇表大小
    d = 5   # 嵌入维度
    context_size = 3
    batch_size = 4
    
    # 创建权重矩阵
    W = torch.randn(V, d, requires_grad=True)
    W_out = torch.randn(d, V, requires_grad=True)
    
    # 创建上下文索引 (每个样本有context_size个上下文词)
    context_indices = torch.randint(0, V, (batch_size, context_size))
    
    # 前向传播计算损失
    loss = cbow_forward(context_indices, W, W_out)
    print("CBOW损失:", loss.item())
    
    # 反向传播
    loss.backward()
    print("W梯度形状:", W.grad.shape)
    print("W_out梯度形状:", W_out.grad.shape)

CBOW损失: 1.6600642204284668
W梯度形状: torch.Size([10, 5])
W_out梯度形状: torch.Size([5, 10])


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ==================== 标准实现 ====================
class MultiHeadAttention(nn.Module):
    """
    多头注意力机制（标准实现）
    
    参数:
        d_model: 模型维度 (默认: 4)
        num_heads: 注意力头数 (默认: 2)
    """
    def __init__(self, d_model=4, num_heads=2):
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度 (4//2=2)
        self.d_v = d_model // num_heads  # 每个头的维度 (4//2=2)
        
        # 线性投影层：Q, K, V
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
        # 最终线性层
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, X, mask=None):
        """
        前向传播
        
        参数:
            X: 输入，形状 (seq_len, batch, d_model)
            mask: 注意力掩码，形状 (batch, seq_len, seq_len) 或 None
        
        返回:
            output: 输出，形状 (seq_len, batch, d_model)
        """
        seq_len, batch, d_model = X.shape
        
        # 1. 线性投影得到 Q, K, V
        # 形状: (seq_len, batch, d_model)
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 2. 重塑为多头形状
        # 将 (seq_len, batch, d_model) -> (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k)
        V = V.view(seq_len, batch, self.num_heads, self.d_v)
        
        # 3. 交换维度以便进行批处理
        # (seq_len, batch, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        Q = Q.permute(1, 2, 0, 3)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)
        
        # 4. 计算缩放点积注意力
        # Q * K^T / sqrt(d_k)
        # (batch, num_heads, seq_len, d_k) @ (batch, num_heads, d_k, seq_len)
        # -> (batch, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # 应用掩码（如果提供）
        if mask is not None:
            # mask 形状: (batch, seq_len, seq_len)
            # 扩展为 (batch, 1, seq_len, seq_len) 以便广播
            scores = scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        
        # Softmax 得到注意力权重
        # (batch, num_heads, seq_len, seq_len)
        attn_weights = F.softmax(scores, dim=-1)
        
        # 注意力加权求和: attn_weights @ V
        # (batch, num_heads, seq_len, seq_len) @ (batch, num_heads, seq_len, d_v)
        # -> (batch, num_heads, seq_len, d_v)
        context = torch.matmul(attn_weights, V)
        
        # 5. 合并多头
        # (batch, num_heads, seq_len, d_v) -> (batch, seq_len, num_heads, d_v)
        context = context.permute(0, 2, 1, 3).contiguous()
        # -> (seq_len, batch, num_heads * d_v) = (seq_len, batch, d_model)
        context = context.view(seq_len, batch, self.d_model)
        
        # 6. 最终线性层
        output = self.W_o(context)
        
        # 保存注意力权重以供可视化
        self.attn_weights = attn_weights
        
        return output


# ==================== 手动实现（详细步骤） ====================
def multi_head_attention_manual(X, W_q, W_k, W_v, W_o, num_heads=2, mask=None):
    """
    多头注意力的手动实现（用于理解具体计算过程）
    
    参数:
        X: 输入，形状 (seq_len, batch, d_model)
        W_q, W_k, W_v, W_o: 权重矩阵
        num_heads: 注意力头数
        mask: 注意力掩码
    
    返回:
        output: 输出，形状 (seq_len, batch, d_model)
        attn_weights: 注意力权重
    """
    seq_len, batch, d_model = X.shape
    d_k = d_model // num_heads
    d_v = d_model // num_heads
    
    print(f"\n=== 多头注意力计算详情 ===")
    print(f"输入形状: {X.shape}")
    print(f"num_heads={num_heads}, d_model={d_model}, d_k={d_k}, d_v={d_v}")
    
    # 1. 线性投影
    print(f"\n1. 线性投影:")
    Q = torch.matmul(X, W_q.T)  # (seq_len, batch, d_model)
    K = torch.matmul(X, W_k.T)  # (seq_len, batch, d_model)
    V = torch.matmul(X, W_v.T)  # (seq_len, batch, d_model)
    print(f"   Q, K, V 形状: {Q.shape}")
    
    # 2. 重塑为多头
    print(f"\n2. 重塑为多头:")
    Q = Q.view(seq_len, batch, num_heads, d_k)
    K = K.view(seq_len, batch, num_heads, d_k)
    V = V.view(seq_len, batch, num_heads, d_v)
    print(f"   Q, K, V 形状 (seq_len, batch, num_heads, d_k): {Q.shape}")
    
    # 3. 转置为 (batch, num_heads, seq_len, d_k)
    Q = Q.permute(1, 2, 0, 3)
    K = K.permute(1, 2, 0, 3)
    V = V.permute(1, 2, 0, 3)
    print(f"\n3. 转置后形状 (batch, num_heads, seq_len, d_k): {Q.shape}")
    
    # 4. 计算缩放点积注意力
    print(f"\n4. 计算注意力分数:")
    # Q @ K^T
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    print(f"   注意力分数形状: {scores.shape}")
    print(f"   示例分数 (head 0, batch 0):\n{scores[0, 0, :, :]}")
    
    # 应用掩码
    if mask is not None:
        scores = scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        print(f"   应用掩码后")
    
    # Softmax
    attn_weights = F.softmax(scores, dim=-1)
    print(f"\n   注意力权重形状: {attn_weights.shape}")
    print(f"   示例权重 (head 0, batch 0):\n{attn_weights[0, 0, :, :]}")
    print(f"   权重和: {attn_weights[0, 0, 0, :].sum().item():.4f}")
    
    # 加权求和
    context = torch.matmul(attn_weights, V)
    print(f"\n5. 加权求和后形状: {context.shape}")
    
    # 5. 合并多头
    context = context.permute(0, 2, 1, 3).contiguous()
    context = context.view(seq_len, batch, d_model)
    print(f"\n6. 合并多头后形状: {context.shape}")
    
    # 6. 最终线性层
    output = torch.matmul(context, W_o.T)
    print(f"\n7. 最终线性层后形状: {output.shape}")
    
    return output, attn_weights


# ==================== 测试代码 ====================
def test_multi_head_attention():
    """测试多头注意力"""
    print("=" * 70)
    print("多头注意力 (Multi-Head Attention) 测试")
    print("=" * 70)
    
    # 设置参数
    d_model = 4
    num_heads = 2
    d_k = d_model // num_heads  # 2
    d_v = d_model // num_heads  # 2
    seq_len = 3
    batch = 2
    
    print(f"\n配置:")
    print(f"  d_model = {d_model}")
    print(f"  num_heads = {num_heads}")
    print(f"  d_k = d_v = {d_k}")
    print(f"  seq_len = {seq_len}")
    print(f"  batch = {batch}")
    
    # 1. 使用标准 PyTorch 模块
    print("\n" + "-" * 70)
    print("方法 1: 使用 nn.Module")
    print("-" * 70)
    
    torch.manual_seed(42)
    model = MultiHeadAttention(d_model, num_heads)
    
    # 创建输入
    X = torch.randn(seq_len, batch, d_model)
    print(f"\n输入 X 形状: {X.shape}")
    print(f"X 值:\n{X}")
    
    # 前向传播
    output = model(X)
    print(f"\n输出形状: {output.shape}")
    print(f"输出值:\n{output}")
    
    # 2. 手动实现（使用相同的权重）
    print("\n" + "-" * 70)
    print("方法 2: 手动实现（详细步骤）")
    print("-" * 70)
    
    # 获取模型的权重
    W_q = model.W_q.weight.data
    W_k = model.W_k.weight.data
    W_v = model.W_v.weight.data
    W_o = model.W_o.weight.data
    
    output_manual, attn_weights = multi_head_attention_manual(
        X, W_q, W_k, W_v, W_o, num_heads
    )
    
    # 验证两种实现的结果是否一致
    print("\n" + "-" * 70)
    print("验证结果")
    print("-" * 70)
    diff = torch.abs(output - output_manual).max().item()
    print(f"两种实现的最大差异: {diff:.6f}")
    if diff < 1e-6:
        print("✅ 结果一致！")
    else:
        print("⚠️ 结果有差异（可能是数值精度问题）")
    
    # 3. 展示注意力权重可视化
    print("\n" + "-" * 70)
    print("注意力权重可视化")
    print("-" * 70)
    
    for h in range(num_heads):
        print(f"\nHead {h} 的注意力权重 (batch 0):")
        weights = attn_weights[0, h, :, :]  # (seq_len, seq_len)
        print(weights)
        print(f"每行和: {weights.sum(dim=1)}")


# ==================== 带掩码的多头注意力 ====================
def test_masked_attention():
    """测试带掩码的多头注意力"""
    print("\n" + "=" * 70)
    print("带掩码的多头注意力测试")
    print("=" * 70)
    
    d_model = 4
    num_heads = 2
    seq_len = 3
    batch = 2
    
    torch.manual_seed(42)
    model = MultiHeadAttention(d_model, num_heads)
    
    X = torch.randn(seq_len, batch, d_model)
    
    # 创建因果掩码（不允许看到未来信息）
    mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
    mask = mask.unsqueeze(0).expand(batch, -1, -1)  # (batch, seq_len, seq_len)
    
    print(f"掩码形状: {mask.shape}")
    print(f"掩码 (batch 0):\n{mask[0]}")
    
    output_masked = model(X, mask)
    print(f"\n带掩码的输出形状: {output_masked.shape}")


# ==================== 逐步计算示例（小规模） ====================
def step_by_step_example():
    """小规模逐步计算示例"""
    print("\n" + "=" * 70)
    print("小规模逐步计算示例 (seq_len=2, batch=1)")
    print("=" * 70)
    
    d_model = 4
    num_heads = 2
    d_k = d_model // num_heads  # 2
    seq_len = 2
    batch = 1
    
    # 创建简单的输入和权重
    torch.manual_seed(0)
    X = torch.randn(seq_len, batch, d_model)
    
    # 创建简单的权重矩阵（便于手动计算）
    W_q = torch.eye(d_model) * 0.5
    W_k = torch.eye(d_model) * 0.5
    W_v = torch.eye(d_model) * 0.5
    W_o = torch.eye(d_model)
    
    print(f"\n输入 X:\n{X.squeeze()}")
    
    # 手动计算
    output, attn_weights = multi_head_attention_manual(
        X, W_q, W_k, W_v, W_o, num_heads
    )
    
    print(f"\n最终输出:\n{output.squeeze()}")
    
    # 显示每个头的影响
    print("\n各头注意力权重:")
    for h in range(num_heads):
        print(f"  Head {h}:\n{attn_weights[0, h, :, :]}")


# ==================== 运行所有测试 ====================
if __name__ == "__main__":
    test_multi_head_attention()
    test_masked_attention()
    step_by_step_example()
    
    print("\n" + "=" * 70)
    print("所有测试完成！")
    print("=" * 70)

多头注意力 (Multi-Head Attention) 测试

配置:
  d_model = 4
  num_heads = 2
  d_k = d_v = 2
  seq_len = 3
  batch = 2

----------------------------------------------------------------------
方法 1: 使用 nn.Module
----------------------------------------------------------------------

输入 X 形状: torch.Size([3, 2, 4])
X 值:
tensor([[[ 1.4451,  0.8564,  2.2181,  0.5232],
         [ 0.3466, -0.1973, -1.0546,  1.2780]],

        [[ 0.7281, -0.7106, -0.6021,  0.9604],
         [ 0.4048, -1.3543, -0.4976,  0.4747]],

        [[-0.1976,  1.2683,  1.2243,  0.0981],
         [ 1.7423, -1.3527,  0.2191,  0.5526]]])

输出形状: torch.Size([3, 2, 4])
输出值:
tensor([[[-0.2483,  0.0803,  0.0932, -0.0505],
         [-0.2407,  0.0626,  0.0929, -0.0319]],

        [[-0.2946,  0.1136,  0.0810, -0.0579],
         [-0.0279, -0.2591,  0.1565,  0.1867]],

        [[-0.0223, -0.2604,  0.1491,  0.1898],
         [-0.0274, -0.2537,  0.1283,  0.1793]]], grad_fn=<UnsafeViewBackward0>)

--------------------------------------------------